In [1]:
path = "Preliminary_24032026(1)/"

In [3]:
import pandas as pd

files = [
    f"qaoa_results_air2.csv",
    f"qaoa_results_new_air2.csv",
    f"qaoa_results_new_mini2.csv",
]


df = pd.concat([pd.read_csv(path + f) for f in files], ignore_index=True)
df.to_csv(path + "merged.csv", index=False)

In [4]:
import pandas as pd

df = pd.read_csv(path + "merged.csv")

df["approximation gap"] = df["result"] / df["m"]
df["Approx guarantee"] = df["approximation gap"] >= 0.6924

df.to_csv(path + "merged.csv", index=False)

In [34]:
import os
import pandas as pd
import matplotlib.pyplot as plt

restrict_length: int | None = None

filter = "[1, 1, 1]"

plots_path = path + f"plots_{filter}/"
plots_path += f"m_restr_{restrict_length}/" if restrict_length is not None else ""
os.makedirs(plots_path, exist_ok=True)

df = pd.read_csv(path + "merged.csv")
df = df[df["parameter_vector"].astype(str) == filter]
df = df[df["m"] <= restrict_length] if restrict_length is not None else df

avg_table = (
    df.groupby(["singlet_injection", "precision", "p"], as_index=False)["approximation gap"]
    .mean()
    .rename(columns={"approximation gap": "avg_approximation_gap"})
)

print(avg_table)
avg_table.to_csv(path + "avg_approximation_gap_by_singlet_precision_p.csv", index=False)

# 1) For fixed singlet_injection + fixed precision: gap vs p
for singlet_value in sorted(df["singlet_injection"].dropna().unique()):
    for precision_value in sorted(df["precision"].dropna().unique()):
        subset = avg_table[
            (avg_table["singlet_injection"] == singlet_value) &
            (avg_table["precision"] == precision_value)
        ].sort_values("p")

        if len(subset) == 0:
            continue

        plt.figure()
        plt.plot(subset["p"], subset["avg_approximation_gap"], marker="o")
        plt.xlabel("p")
        plt.ylabel("Average approximation gap")
        plt.title(f"gap vs p | precision={precision_value} | singlet_injection={singlet_value}")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(plots_path + f"gap_vs_p_precision_{precision_value}_singlet_{singlet_value}.png")
        plt.close()

# 2) For fixed singlet_injection + fixed p: gap vs precision
for singlet_value in sorted(df["singlet_injection"].dropna().unique()):
    for p_value in sorted(df["p"].dropna().unique()):
        subset = avg_table[
            (avg_table["singlet_injection"] == singlet_value) &
            (avg_table["p"] == p_value)
        ].sort_values("precision", ascending=False)

        if len(subset) == 0:
            continue

        plt.figure()
        plt.plot(subset["precision"], subset["avg_approximation_gap"], marker="o")
        plt.gca().invert_xaxis()
        plt.xlabel("precision")
        plt.ylabel("Average approximation gap")
        plt.title(f"gap vs precision | p={p_value} | singlet_injection={singlet_value}")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(plots_path + f"gap_vs_precision_p_{p_value}_singlet_{singlet_value}.png")
        plt.close()

# 3) Compare singlet_injection True/False for same precision: gap vs p
for precision_value in sorted(df["precision"].dropna().unique()):
    plt.figure()
    plotted = False

    for singlet_value in sorted(df["singlet_injection"].dropna().unique()):
        subset = avg_table[
            (avg_table["precision"] == precision_value) &
            (avg_table["singlet_injection"] == singlet_value)
        ].sort_values("p")

        if len(subset) == 0:
            continue

        plt.plot(subset["p"], subset["avg_approximation_gap"], marker="o", label=f"singlet_injection={singlet_value}")
        plotted = True

    if plotted:
        plt.xlabel("p")
        plt.ylabel("Average approximation gap")
        plt.title(f"gap vs p | precision={precision_value}")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(plots_path + f"compare_singlet_gap_vs_p_precision_{precision_value}.png")
        plt.close()
    else:
        plt.close()

# 4) Compare singlet_injection True/False for same p: gap vs precision
avg_table["precision"] = pd.to_numeric(avg_table["precision"])

for p_value in sorted(df["p"].dropna().unique()):
    plt.figure()
    plotted = False

    precisions_for_plot = sorted(
        avg_table.loc[avg_table["p"] == p_value, "precision"].dropna().unique(),
        reverse=True
    )
    x_positions = list(range(len(precisions_for_plot)))
    x_map = {prec: i for i, prec in enumerate(precisions_for_plot)}

    for singlet_value in sorted(df["singlet_injection"].dropna().unique()):
        subset = avg_table[
            (avg_table["p"] == p_value) &
            (avg_table["singlet_injection"] == singlet_value)
        ].copy()

        if len(subset) == 0:
            continue

        subset["precision"] = pd.to_numeric(subset["precision"])
        subset = subset.sort_values("precision", ascending=False)

        plt.plot(
            [x_map[prec] for prec in subset["precision"]],
            subset["avg_approximation_gap"],
            marker="o",
            label=f"singlet_injection={singlet_value}"
        )
        plotted = True

    if plotted:
        plt.xlabel("precision")
        plt.ylabel("Average approximation gap")
        plt.title(f"gap vs precision | p={p_value}")
        plt.xticks(x_positions, [str(x) for x in precisions_for_plot])
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(plots_path + f"compare_singlet_gap_vs_precision_p_{p_value}.png")
        plt.close()
    else:
        plt.close()



    singlet_injection  precision  p  avg_approximation_gap
0               False      0.001  1           2.777778e-01
1               False      0.010  1           5.070914e-02
2               False      0.100  1           4.380094e-17
3               False      0.100  2           3.361261e-01
4               False      0.200  1           4.110248e-17
5               False      0.200  2           2.333212e-01
6               False      0.500  1           2.309660e-17
7               False      0.500  2           5.077839e-17
8               False      0.500  3           4.054887e-01
9                True      0.001  1           3.125000e-01
10               True      0.010  1           5.444527e-01
11               True      0.100  1           5.543792e-01
12               True      0.100  2           5.338224e-01
13               True      0.200  1           5.543642e-01
14               True      0.200  2           5.534438e-01
15               True      0.500  1           5.542753e-

In [38]:
import pandas as pd
from itertools import product
filter = "[1, 1, 1]"

df = pd.read_csv(path + "merged.csv")
df = df[df["parameter_vector"].astype(str) == filter]

m_values = sorted(df["m"].dropna().unique())
p_values = sorted(df["p"].dropna().unique())
precision_values = sorted(df["precision"].dropna().unique())

grouped = df.groupby(["m", "p", "precision"])["singlet_injection"].agg(set).reset_index()

missing = []
for _, row in grouped.iterrows():
    present = row["singlet_injection"]
    if True not in present:
        missing.append({
            "m": row["m"],
            "p": row["p"],
            "precision": row["precision"],
            "missing_singlet_injection": True
        })
    if False not in present:
        missing.append({
            "m": row["m"],
            "p": row["p"],
            "precision": row["precision"],
            "missing_singlet_injection": False
        })

missing_df = pd.DataFrame(missing)
print(missing_df)
missing_df.to_csv(path + f"missing_benchmarks_{filter}.csv", index=False)

    m  p  precision  missing_singlet_injection
0   3  1      0.001                       True
1   8  2      0.100                       True
2  10  1      0.010                       True


In [39]:
df_all = pd.read_csv(path + "merged.csv")
df_all["precision"] = pd.to_numeric(df_all["precision"])
df_all = df_all[df_all["m"] <= restrict_length] if restrict_length is not None else df_all


avg_all = (
    df_all.groupby(["parameter_vector", "singlet_injection", "precision", "p"], as_index=False)["approximation gap"]
    .mean()
    .rename(columns={"approximation gap": "avg_approximation_gap"})
)

for param in [filter]:
    sub_param = avg_all[avg_all["parameter_vector"].astype(str) == param].copy()

    fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
    ax.set_facecolor("white")

    singlet_colors = {
        False: ["lightsteelblue", "cornflowerblue", "royalblue", "blue", "navy"],
        True:  ["wheat", "gold", "orange", "darkorange", "orangered"]
    }
    singlet_linestyles = {
        False: "-",
        True: "--"
    }

    precisions_for_plot = sorted(sub_param["precision"].dropna().unique(), reverse=True)
    x_positions = list(range(len(precisions_for_plot)))
    x_map = {prec: i for i, prec in enumerate(precisions_for_plot)}

    for singlet_value in sorted(sub_param["singlet_injection"].dropna().unique()):
        p_values = sorted(sub_param[sub_param["singlet_injection"] == singlet_value]["p"].dropna().unique())

        for i, p_value in enumerate(p_values):
            subset = sub_param[
                (sub_param["singlet_injection"] == singlet_value) &
                (sub_param["p"] == p_value)
            ].sort_values("precision", ascending=False)

            if len(subset) == 0:
                continue

            color_list = singlet_colors[singlet_value]
            color = color_list[min(i, len(color_list) - 1)]

            ax.plot(
                [x_map[prec] for prec in subset["precision"]],
                subset["avg_approximation_gap"],
                marker="o",
                linestyle=singlet_linestyles[singlet_value],
                color=color,
                label=f"p={p_value}, singlet={singlet_value}"
            )

    ax.set_xlabel("precision", color="black")
    ax.set_ylabel("Average approximation gap", color="black")
    ax.set_title(f"gap vs precision | parameter_vector={param}", color="black")
    ax.set_xticks(x_positions)
    ax.set_xticklabels([str(x) for x in precisions_for_plot], color="black")
    ax.tick_params(axis="y", colors="black")

    for spine in ax.spines.values():
        spine.set_color("black")

    ax.grid(True, color="lightgray", alpha=0.6)

    legend = ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), facecolor="white", edgecolor="black")
    for text in legend.get_texts():
        text.set_color("black")

    fig.subplots_adjust(right=0.72)
    fig.savefig(
        plots_path + "overview_all_p_by_singlet.png",
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close(fig)

for param in [filter]:
    sub_param = avg_all[avg_all["parameter_vector"].astype(str) == param].copy()

    fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
    ax.set_facecolor("white")

    singlet_colors = {
        False: ["lightsteelblue", "cornflowerblue", "royalblue", "blue", "navy"],
        True:  ["wheat", "gold", "orange", "darkorange", "orangered"]
    }
    singlet_linestyles = {
        False: "-",
        True: "--"
    }

    for singlet_value in sorted(sub_param["singlet_injection"].dropna().unique()):
        precisions = sorted(sub_param[sub_param["singlet_injection"] == singlet_value]["precision"].dropna().unique())

        for i, precision_value in enumerate(precisions):
            subset = sub_param[
                (sub_param["singlet_injection"] == singlet_value) &
                (sub_param["precision"] == precision_value)
            ].sort_values("p")

            if len(subset) == 0:
                continue

            color_list = singlet_colors[singlet_value]
            color = color_list[min(i, len(color_list) - 1)]

            ax.plot(
                subset["p"],
                subset["avg_approximation_gap"],
                marker="o",
                linestyle=singlet_linestyles[singlet_value],
                color=color,
                label=f"precision={precision_value}, singlet={singlet_value}"
            )

    ax.set_xlabel("p", color="black")
    ax.set_ylabel("Average approximation gap", color="black")
    ax.set_title(f"gap vs p | parameter_vector={param}", color="black")

    ax.tick_params(axis="both", colors="black")

    for spine in ax.spines.values():
        spine.set_color("black")

    ax.grid(True, color="lightgray", alpha=0.6)

    legend = ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), facecolor="white", edgecolor="black")
    for text in legend.get_texts():
        text.set_color("black")

    fig.subplots_adjust(right=0.72)
    fig.savefig(
        plots_path + f"overview_all_precision_by_singlet.png",
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close(fig)

In [40]:
"""Granular plots per m to check monotonicity over especially increased precision:"""


df_all = pd.read_csv(path + "merged.csv")
df_all["precision"] = pd.to_numeric(df_all["precision"])
df_all = df_all[df_all["m"] <= restrict_length] if restrict_length is not None else df_all

avg_m = (
    df_all.groupby(["parameter_vector", "m", "singlet_injection", "precision", "p"], as_index=False)["approximation gap"]
    .mean()
    .rename(columns={"approximation gap": "avg_approximation_gap"})
)

for param in [filter]:
    sub_param = avg_m[avg_m["parameter_vector"].astype(str) == param].copy()

    for m_value in sorted(sub_param["m"].dropna().unique()):
        sub_m = sub_param[sub_param["m"] == m_value].copy()

        # overview: gap vs precision, separate per m
        fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
        ax.set_facecolor("white")

        singlet_colors = {
            False: ["lightsteelblue", "cornflowerblue", "royalblue", "blue", "navy"],
            True:  ["wheat", "gold", "orange", "darkorange", "orangered"]
        }
        singlet_linestyles = {
            False: "-",
            True: "--"
        }

        precisions_for_plot = sorted(sub_m["precision"].dropna().unique(), reverse=True)
        x_positions = list(range(len(precisions_for_plot)))
        x_map = {prec: i for i, prec in enumerate(precisions_for_plot)}

        for singlet_value in sorted(sub_m["singlet_injection"].dropna().unique()):
            p_values = sorted(sub_m[sub_m["singlet_injection"] == singlet_value]["p"].dropna().unique())

            for i, p_value in enumerate(p_values):
                subset = sub_m[
                    (sub_m["singlet_injection"] == singlet_value) &
                    (sub_m["p"] == p_value)
                ].sort_values("precision", ascending=False)

                if len(subset) == 0:
                    continue

                color_list = singlet_colors[singlet_value]
                color = color_list[min(i, len(color_list) - 1)]

                ax.plot(
                    [x_map[prec] for prec in subset["precision"]],
                    subset["avg_approximation_gap"],
                    marker="o",
                    linestyle=singlet_linestyles[singlet_value],
                    color=color,
                    label=f"p={p_value}, singlet={singlet_value}"
                )

        ax.set_xlabel("precision", color="black")
        ax.set_ylabel("Average approximation gap", color="black")
        ax.set_title(f"gap vs precision | parameter_vector={param} | m={m_value}", color="black")
        ax.set_xticks(x_positions)
        ax.set_xticklabels([str(x) for x in precisions_for_plot], color="black")
        ax.tick_params(axis="y", colors="black")

        for spine in ax.spines.values():
            spine.set_color("black")

        ax.grid(True, color="lightgray", alpha=0.6)

        legend = ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), facecolor="white", edgecolor="black")
        for text in legend.get_texts():
            text.set_color("black")

        fig.subplots_adjust(right=0.72)
        fig.savefig(
            plots_path + f"overview_all_p_by_singlet_m_{m_value}.png",
            bbox_inches="tight",
            facecolor="white"
        )
        plt.close(fig)

        # overview: gap vs p, separate per m
        fig, ax = plt.subplots(figsize=(14, 6), facecolor="white")
        ax.set_facecolor("white")

        for singlet_value in sorted(sub_m["singlet_injection"].dropna().unique()):
            precisions = sorted(sub_m[sub_m["singlet_injection"] == singlet_value]["precision"].dropna().unique())

            for i, precision_value in enumerate(precisions):
                subset = sub_m[
                    (sub_m["singlet_injection"] == singlet_value) &
                    (sub_m["precision"] == precision_value)
                ].sort_values("p")

                if len(subset) == 0:
                    continue

                color_list = singlet_colors[singlet_value]
                color = color_list[min(i, len(color_list) - 1)]

                ax.plot(
                    subset["p"],
                    subset["avg_approximation_gap"],
                    marker="o",
                    linestyle=singlet_linestyles[singlet_value],
                    color=color,
                    label=f"precision={precision_value}, singlet={singlet_value}"
                )

        ax.set_xlabel("p", color="black")
        ax.set_ylabel("Average approximation gap", color="black")
        ax.set_title(f"gap vs p | parameter_vector={param} | m={m_value}", color="black")
        ax.tick_params(axis="both", colors="black")

        for spine in ax.spines.values():
            spine.set_color("black")

        ax.grid(True, color="lightgray", alpha=0.6)

        legend = ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), facecolor="white", edgecolor="black")
        for text in legend.get_texts():
            text.set_color("black")

        fig.subplots_adjust(right=0.72)
        fig.savefig(
            plots_path + f"overview_all_precision_by_singlet_m_{m_value}.png",
            bbox_inches="tight",
            facecolor="white"
        )
        plt.close(fig)

In [35]:
import pandas as pd
from itertools import product

df = pd.read_csv(path + "merged.csv")

wanted = ["[0, 0, 1]", "[1, 1, 1]"]
pairs = sorted(set(zip(df["p"], df["precision"])))
done = set(zip(df["p"], df["precision"], df["parameter_vector"].astype(str), df["singlet_injection"]))

checklist = pd.DataFrame([
    {
        "setting": f"p={p}, precision={precision}",
        "[0,0,1] | False": "x" if (p, precision, "[0, 0, 1]", False) in done else "",
        "[0,0,1] | True":  "x" if (p, precision, "[0, 0, 1]", True)  in done else "",
        "[1,1,1] | False": "x" if (p, precision, "[1, 1, 1]", False) in done else "",
        "[1,1,1] | True":  "x" if (p, precision, "[1, 1, 1]", True)  in done else "",
    }
    for p, precision in pairs
])

print(checklist)
checklist.to_csv(path + "benchmark_checklist.csv", index=False)

                setting [0,0,1] | False [0,0,1] | True [1,1,1] | False  \
0  p=1, precision=0.001               x              x                   
1   p=1, precision=0.01               x              x               x   
2    p=1, precision=0.1               x              x               x   
3    p=1, precision=0.2               x              x               x   
4    p=1, precision=0.5               x              x               x   
5    p=2, precision=0.1               x              x               x   
6    p=2, precision=0.2               x              x               x   
7    p=2, precision=0.5               x              x               x   
8    p=3, precision=0.5               x              x               x   

  [1,1,1] | True  
0                 
1                 
2              x  
3              x  
4              x  
5              x  
6              x  
7              x  
8              x  


In [39]:
import pandas as pd

df = pd.read_csv(path + "merged.csv")
df["parameter_vector"] = df["parameter_vector"].astype(str)

def m_range(param, singlet, p, precision):
    vals = sorted(df[
        (df["parameter_vector"] == param) &
        (df["singlet_injection"] == singlet) &
        (df["p"] == p) &
        (df["precision"] == precision)
    ]["m"].dropna().astype(int).unique())
    return f"[{vals[0]}, {vals[-1]}]" if vals else ""

pairs = sorted(set(zip(df["p"], df["precision"])))

out = pd.DataFrame([{
    "setting": f"p={p}, precision={precision}",
    "[0,0,1] | False": m_range("[0, 0, 1]", False, p, precision),
    "[0,0,1] | True":  m_range("[0, 0, 1]", True,  p, precision),
    "[1,1,1] | False": m_range("[1, 1, 1]", False, p, precision),
    "[1,1,1] | True":  m_range("[1, 1, 1]", True,  p, precision),
} for p, precision in pairs])

print(out)
out.to_csv(path + "benchmark_m_ranges.csv", index=False)

                setting [0,0,1] | False [0,0,1] | True [1,1,1] | False  \
0  p=1, precision=0.001          [1, 5]         [1, 4]                   
1   p=1, precision=0.01         [1, 10]        [1, 10]         [1, 10]   
2    p=1, precision=0.1         [1, 12]        [1, 13]         [1, 10]   
3    p=1, precision=0.2         [1, 10]        [1, 10]         [1, 10]   
4    p=1, precision=0.5         [1, 10]        [1, 10]         [1, 10]   
5    p=2, precision=0.1          [1, 7]         [1, 7]          [1, 7]   
6    p=2, precision=0.2          [1, 9]         [1, 9]          [1, 9]   
7    p=2, precision=0.5         [1, 10]        [1, 10]         [1, 10]   
8    p=3, precision=0.5          [1, 9]         [1, 8]          [1, 8]   

  [1,1,1] | True  
0                 
1                 
2        [1, 10]  
3        [1, 10]  
4        [1, 10]  
5         [1, 6]  
6         [1, 8]  
7        [1, 10]  
8         [1, 8]  
